# Module 5.3 — MMR: Maximal Marginal Relevance

MMR balances **relevance** (to the query) and **diversity** (among results), avoiding redundant chunks.

$$\text{MMR}(d_i) = \lambda \cdot \text{Sim}(d_i, q) - (1-\lambda) \cdot \max_{d_j \in S} \text{Sim}(d_i, d_j)$$

- `lambda_mult=1.0` → pure relevance (same as similarity search)
- `lambda_mult=0.0` → pure diversity
- `fetch_k` → candidate pool size before MMR re-ranking
- `k` → final returned documents

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Corpus with intentional redundancy
docs = [
    "Paris is the capital of France and a major European city.",
    "Paris, France's capital, is famous for the Eiffel Tower.",
    "The city of Paris is located in north-central France.",
    "London is the capital of England.",
    "Berlin is the capital of Germany.",
    "Rome is the capital of Italy.",
]

vs = Chroma.from_documents(
    [Document(page_content=d) for d in docs],
    embeddings, collection_name="mmr_demo"
)

query = "Tell me about European capitals"

# ── Standard similarity search ────────────────────────────────────────────────
std = vs.similarity_search(query, k=4)
print("Standard similarity_search (k=4):")
for d in std:
    print(f"  • {d.page_content}")

# ── MMR (diverse) ─────────────────────────────────────────────────────────────
mmr = vs.max_marginal_relevance_search(query, k=4, fetch_k=6, lambda_mult=0.5)
print("\nMMR (lambda=0.5, fetch_k=6, k=4):")
for d in mmr:
    print(f"  • {d.page_content}")
